<a href="https://colab.research.google.com/github/bongsuyeong/hdac-smoking-abdominal-obesity/blob/main/10_table1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# pyreadstat 라이브러리가 없다면 자동으로 설치합니다
!pip install pyreadstat -q

import pandas as pd
import numpy as np
import pyreadstat
from google.colab import drive

# Google Drive 연동
drive.mount('/content/drive')

# 파일 경로 (파일을 우클릭해서 복사한 경로를 여기에 붙여넣으세요)
file_path = '/content/drive/MyDrive/hdac_data/hn24.sav'

# 원시자료 불러오기 (메타데이터와 데이터프레임 분리)
raw_df, meta = pyreadstat.read_sav(file_path)
print("원시자료 행/열 크기:", raw_df.shape)

Mounted at /content/drive
원시자료 행/열 크기: (6997, 813)


In [ ]:
# [셀 2] 변수 추출 및 재코딩 (9주차 프로토콜 반영)
vars_needed = ['year', 'kstrata', 'psu', 'wt_itvex', 'age', 'sex', 'BS3_1', 'BS1_1', 'HE_wc', 'incm', 'edu', 'BD1_11', 'pa_aerobic']
df = raw_df[vars_needed].copy()

# 1. 노출 정의: BS3_1 (1, 2 -> 현재흡연 1 / 3, 8 -> 비흡연 0 / 9 및 결측 -> 제외)
def recode_smoking(row):
    bs3 = row['BS3_1']
    if bs3 in [1, 2]:
        return 1  # 현재흡연군
    elif bs3 in [3, 8]:
        return 0  # 비흡연군 (과거흡연 + 평생 비흡연/비해당)
    else:
        return np.nan

df['smoke_grp'] = df.apply(recode_smoking, axis=1)

# 2. 결과 정의: 복부비만 (남성 >= 90cm, 여성 >= 85cm)
def recode_abdominal_obesity(row):
    wc = row['HE_wc']
    sex = row['sex']
    if pd.isna(wc):
        return np.nan
    if sex == 1 and wc >= 90:
        return 1
    elif sex == 2 and wc >= 85:
        return 1
    else:
        return 0

df['abdominal_obesity'] = df.apply(recode_abdominal_obesity, axis=1)

# 3. 음주 공변량 재코딩 (BD1_11)
def recode_alcohol(val):
    if val == 1 or val == 8:
        return 0
    elif val in [2, 3, 4, 5, 6]:
        return 1
    else:
        return np.nan

df['alc_recode'] = df['BD1_11'].apply(recode_alcohol)

# 4. 유산소 신체활동 실천율 (pa_aerobic)
df['pa_recode'] = df['pa_aerobic'].apply(lambda x: x if x in [0, 1] else np.nan)

print("변수 재코딩 완료!")

변수 재코딩 완료!


In [ ]:
# [셀 3] B-1 대상자 흐름도 인원 계산 및 검증
# 1단계: 전체 참여자
step1 = len(df)

# 2단계: 만 20세 이상 성인
step2_df = df[df['age'] >= 20]
step2 = len(step2_df)
step2_exc = step1 - step2

# 3단계: 노출 정의 가능자 (BS3_1이 1, 2, 3, 8인 사람)
step3_df = step2_df[step2_df['smoke_grp'].notna()]
step3 = len(step3_df)
step2_3_exc = step2 - step3

# 4단계: 최종 분석 대상 (결과 허리둘레 HE_wc까지 있는 사람)
final_df = step3_df[step3_df['HE_wc'].notna()].copy()
final_n = len(final_df)
step3_4_exc = step3 - final_n

# 분석 대상 표시변수(in_analysis) 생성
df['in_analysis'] = 0
df.loc[final_df.index, 'in_analysis'] = 1

# 최종 흡연군 / 비흡연군 인원
n_smoke = (final_df['smoke_grp'] == 1).sum()
n_nonsmoke = (final_df['smoke_grp'] == 0).sum()

print("=== B-1 대상자 흐름도 인원 검증 ===")
print(f"1단계 (전체 참여자): {step1}")
print(f"2단계 (20세 이상 성인): {step2} (제외: {step2_exc})")
print(f"3단계 (노출 정의 가능): {step3} (제외: {step2_3_exc})")
print(f"4단계 (최종 분석 대상): {final_n} (제외: {step3_4_exc})")
print(f" -> 현재흡연군: {n_smoke}, 비흡연군: {n_nonsmoke}")

=== B-1 대상자 흐름도 인원 검증 ===
1단계 (전체 참여자): 6997
2단계 (20세 이상 성인): 6002 (제외: 995)
3단계 (노출 정의 가능): 5784 (제외: 218)
4단계 (최종 분석 대상): 5629 (제외: 155)
 -> 현재흡연군: 765, 비흡연군: 4864


In [ ]:
# [셀 4] Table 1 주요 변수별 빈도 및 평균 확인 (비가중 n 기준)
analysis_df = df[df['in_analysis'] == 1].copy()

print("=== 1. 복부비만 유병 빈도 ===")
print(pd.crosstab(analysis_df['smoke_grp'], analysis_df['abdominal_obesity'], margins=True))

print("\n=== 2. 허리둘레 평균 (비가중) ===")
print(analysis_df.groupby('smoke_grp')['HE_wc'].mean())

print("\n=== 3. 성별 빈도 ===")
print(pd.crosstab(analysis_df['smoke_grp'], analysis_df['sex'], margins=True))

print("\n=== 4. 연령 평균 (비가중) ===")
print(analysis_df.groupby('smoke_grp')['age'].mean())

=== 1. 복부비만 유병 빈도 ===
abdominal_obesity   0.0   1.0   All
smoke_grp                          
0.0                3144  1720  4864
1.0                 459   306   765
All                3603  2026  5629

=== 2. 허리둘레 평균 (비가중) ===
smoke_grp
0.0    83.259046
1.0    86.987843
Name: HE_wc, dtype: float64

=== 3. 성별 빈도 ===
sex         1.0   2.0   All
smoke_grp                  
0.0        1774  3090  4864
1.0         636   129   765
All        2410  3219  5629

=== 4. 연령 평균 (비가중) ===
smoke_grp
0.0    54.512541
1.0    50.539869
Name: age, dtype: float64


In [ ]:
# [셀 5] 설계 기반 추정 함수 (가중치 wt_itvex, 층 kstrata, 집락 psu)
# 행을 지우지 않고 전체 6,997행에서 도메인 지시변수(in_analysis x 흡연군)로 분석한다.
from scipy import stats

W, H, C = 'wt_itvex', 'kstrata', 'psu'

def design_ratio(y, dom, data=None):
    """도메인 dom(0/1) 안에서 y의 가중평균(0/1 변수면 비율)과 층·집락 선형화 SE를 반환.
    y가 결측인 행은 도메인에서 빠진다. 층·집락 구조는 전체 행 기준으로 유지한다."""
    data = df if data is None else data
    y = np.asarray(y, dtype=float)
    d = np.asarray(dom, dtype=float) * (~np.isnan(y))
    y0 = np.where(d > 0, y, 0.0)
    w = data[W].to_numpy(dtype=float)
    wd = w * d
    den = wd.sum()
    r = (wd * y0).sum() / den
    z = wd * (y0 - r) / den                      # 선형화 영향값
    tmp = pd.DataFrame({'h': data[H].to_numpy(), 'c': data[C].to_numpy(), 'z': z}) \
            .groupby(['h', 'c'])['z'].sum().reset_index()
    var = 0.0
    for _, g in tmp.groupby('h'):
        n = len(g)
        if n > 1:                                 # 집락이 1개뿐인 층은 기여 없음
            var += n / (n - 1) * ((g['z'] - g['z'].mean()) ** 2).sum()
    return r, float(np.sqrt(var)), int(d.sum())

# 도메인: 전체(분석 대상) / 현재흡연군 / 비흡연군
doms = {
    '전체': (df['in_analysis'] == 1).astype(int).to_numpy(),
    '현재흡연군': ((df['in_analysis'] == 1) & (df['smoke_grp'] == 1)).astype(int).to_numpy(),
    '비흡연군(과거흡연 포함)': ((df['in_analysis'] == 1) & (df['smoke_grp'] == 0)).astype(int).to_numpy(),
}

# 설계 확인
print("전체 표본 층 수:", df[H].nunique(), "/ 집락 수:", df.groupby([H, C]).ngroups)
print("최종 분석 대상에 포함된 집락 수:", df[df['in_analysis'] == 1].groupby([H, C]).ngroups)

전체 표본 층 수: 27 / 집락 수: 192
최종 분석 대상에 포함된 집락 수: 191


In [ ]:
# [셀 6] Table 1 자동 생성 (B-2 가중 / B-3 비가중, 동일한 분석 대상·재코딩 사용)
specs = [
    ('복부비만 (남 ≥90cm, 여 ≥85cm)', 'abdominal_obesity', 'cat', [(1, '예')]),
    ('허리둘레, cm', 'HE_wc', 'cont', None),
    ('성별', 'sex', 'cat', [(1, '남자'), (2, '여자')]),
    ("연령, 세 (80은 '80세 이상')", 'age', 'cont', None),
    ('소득 4분위수(개인)', 'incm', 'cat', [(1, '하'), (2, '중하'), (3, '중상'), (4, '상')]),
    ('교육수준', 'edu', 'cat', [(1, '초졸 이하'), (2, '중졸'), (3, '고졸'), (4, '대졸 이상')]),
    ('최근 1년 음주 (BD1_11 2–6)', 'alc_recode', 'cat', [(1, '음주'), (0, '비음주')]),
    ('유산소 신체활동 실천', 'pa_recode', 'cat', [(1, '실천'), (0, '미실천')]),
]
IND = '\u3000'  # 들여쓰기용 전각 공백 (문서와 동일)
cols = list(doms.keys())

def build_table(weighted):
    ns = {k: int(v.sum()) for k, v in doms.items()}
    header = ['변수'] + [f"{k} (n = {ns[k]:,})" for k in cols]
    rows = []
    for label, var, kind, cats in specs:
        x = df[var].to_numpy(dtype=float)
        if kind == 'cont':
            cells = []
            for k in cols:
                d = doms[k].astype(bool)
                if weighted:
                    m, se, _ = design_ratio(x, doms[k])
                    cells.append(f"{m:.1f} ({se:.2f})")
                else:
                    v = x[d & ~np.isnan(x)]
                    cells.append(f"{v.mean():.1f} ({v.std(ddof=1):.2f})")
            rows.append([f"{label} — 평균 ({'SE' if weighted else 'SD'})"] + cells)
        else:
            rows.append([label] + [''] * 3)
            for code_, cname in cats:
                cells = []
                for k in cols:
                    d = doms[k].astype(bool)
                    valid = d & ~np.isnan(x)
                    n_cat = int((valid & (x == code_)).sum())
                    if weighted:
                        p, _, _ = design_ratio((x == code_).astype(float) + 0 * x, doms[k])
                        p *= 100
                    else:
                        p = 100 * n_cat / valid.sum()
                    cells.append(f"{n_cat:,} ({p:.1f})")
                rows.append([IND + cname] + cells)
            miss = [int((doms[k].astype(bool) & np.isnan(x)).sum()) for k in cols]
            if any(miss):
                rows.append([IND + '결측'] + [str(m) for m in miss])
    return header, rows

def to_md(header, rows):
    out = ['| ' + ' | '.join(header) + ' |', '| :--- | ---: | ---: | ---: |']
    out += ['| ' + ' | '.join(r) + ' |' for r in rows]
    return '\n'.join(out)

hdr_w, rows_w = build_table(weighted=True)
hdr_u, rows_u = build_table(weighted=False)
md_w, md_u = to_md(hdr_w, rows_w), to_md(hdr_u, rows_u)

print("=== B-2. Table 1 (가중 %, 가중 평균(SE); n은 비가중) ===")
print(md_w)
print("\n=== B-3. 비가중 표 (비가중 %, 평균(SD)) ===")
print(md_u)

=== B-2. Table 1 (가중 %, 가중 평균(SE); n은 비가중) ===
| 변수 | 전체 (n = 5,629) | 현재흡연군 (n = 765) | 비흡연군(과거흡연 포함) (n = 4,864) |
| :--- | ---: | ---: | ---: |
| 복부비만 (남 ≥90cm, 여 ≥85cm) |  |  |  |
| 　예 | 2,026 (35.5) | 306 (40.3) | 1,720 (34.6) |
| 허리둘레, cm — 평균 (SE) | 84.0 (0.21) | 87.4 (0.42) | 83.4 (0.21) |
| 성별 |  |  |  |
| 　남자 | 2,410 (50.0) | 636 (87.3) | 1,774 (42.9) |
| 　여자 | 3,219 (50.0) | 129 (12.7) | 3,090 (57.1) |
| 연령, 세 (80은 '80세 이상') — 평균 (SE) | 49.8 (0.47) | 47.2 (0.69) | 50.3 (0.50) |
| 소득 4분위수(개인) |  |  |  |
| 　하 | 1,383 (24.2) | 234 (28.1) | 1,149 (23.5) |
| 　중하 | 1,407 (25.1) | 191 (26.7) | 1,216 (24.8) |
| 　중상 | 1,413 (25.4) | 177 (24.3) | 1,236 (25.6) |
| 　상 | 1,410 (25.3) | 161 (20.9) | 1,249 (26.1) |
| 　결측 | 16 | 2 | 14 |
| 교육수준 |  |  |  |
| 　초졸 이하 | 934 (11.5) | 67 (5.8) | 867 (12.6) |
| 　중졸 | 584 (8.1) | 87 (9.0) | 497 (7.9) |
| 　고졸 | 1,845 (34.0) | 325 (42.9) | 1,520 (32.3) |
| 　대졸 이상 | 2,260 (46.5) | 284 (42.4) | 1,976 (47.2) |
| 　결측 | 6 | 2 | 4 |
| 최근 1년 음주 (BD1_11 2–6)

In [ ]:
# [셀 7] B-3 비교표: 비가중 대 가중 (전체 열 기준)
def pct_by_cat(var, cats, dom):
    x = df[var].to_numpy(dtype=float)
    valid = dom.astype(bool) & ~np.isnan(x)
    out = {}
    for code_, name in cats:
        wt = design_ratio((x == code_).astype(float) + 0 * x, dom)[0] * 100
        un = 100 * (valid & (x == code_)).sum() / valid.sum()
        out[name] = (un, wt, wt - un)
    return out

dom_all = doms['전체']
diff_rows = []
for label, var, kind, cats in specs:
    if kind == 'cat' and var != 'abdominal_obesity':
        for name, (un, wt, df_) in pct_by_cat(var, cats, dom_all).items():
            diff_rows.append((label, name, f"{un:.1f}%", f"{wt:.1f}%", f"{df_:+.1f}%p", abs(df_)))
    elif kind == 'cont':
        x = df[var].to_numpy(dtype=float)
        un_v = x[dom_all.astype(bool) & ~np.isnan(x)]
        wt_m = design_ratio(x, dom_all)[0]
        sd = (wt_m - un_v.mean()) / un_v.std(ddof=1) * 100
        diff_rows.append((label, '평균', f"{un_v.mean():.1f}", f"{wt_m:.1f}", f"{sd:+.1f} (표준화)", abs(sd)))

print("| 변수 | 구분 | 비가중 | 가중 | 차이 |\n| :--- | :--- | ---: | ---: | ---: |")
for r in diff_rows:
    print("| " + " | ".join(r[:5]) + " |")

print("\n변수별 최대 차이:")
best = {}
for r in diff_rows:
    best[r[0]] = max(best.get(r[0], 0), r[5])
for k, v in sorted(best.items(), key=lambda kv: -kv[1]):
    print(f"  {k}: {v:.1f}")

| 변수 | 구분 | 비가중 | 가중 | 차이 |
| :--- | :--- | ---: | ---: | ---: |
| 허리둘레, cm | 평균 | 83.8 | 84.0 | +2.4 (표준화) |
| 성별 | 남자 | 42.8% | 50.0% | +7.2%p |
| 성별 | 여자 | 57.2% | 50.0% | -7.2%p |
| 연령, 세 (80은 '80세 이상') | 평균 | 54.0 | 49.8 | -24.6 (표준화) |
| 소득 4분위수(개인) | 하 | 24.6% | 24.2% | -0.4%p |
| 소득 4분위수(개인) | 중하 | 25.1% | 25.1% | +0.0%p |
| 소득 4분위수(개인) | 중상 | 25.2% | 25.4% | +0.2%p |
| 소득 4분위수(개인) | 상 | 25.1% | 25.3% | +0.1%p |
| 교육수준 | 초졸 이하 | 16.6% | 11.5% | -5.1%p |
| 교육수준 | 중졸 | 10.4% | 8.1% | -2.3%p |
| 교육수준 | 고졸 | 32.8% | 34.0% | +1.2%p |
| 교육수준 | 대졸 이상 | 40.2% | 46.5% | +6.3%p |
| 최근 1년 음주 (BD1_11 2–6) | 음주 | 70.7% | 75.3% | +4.6%p |
| 최근 1년 음주 (BD1_11 2–6) | 비음주 | 29.3% | 24.7% | -4.6%p |
| 유산소 신체활동 실천 | 실천 | 43.8% | 47.3% | +3.4%p |
| 유산소 신체활동 실천 | 미실천 | 56.2% | 52.7% | -3.4%p |

변수별 최대 차이:
  연령, 세 (80은 '80세 이상'): 24.6
  성별: 7.2
  교육수준: 6.3
  최근 1년 음주 (BD1_11 2–6): 4.6
  유산소 신체활동 실천: 3.4
  허리둘레, cm: 2.4
  소득 4분위수(개인): 0.4


In [ ]:
# [셀 8] 문서의 설명 문장에 쓰인 보조 수치 (평균 가중치, 연령대별 구성, 왜도, 공식 정의 비교)
a = df[df['in_analysis'] == 1].copy()

print("=== 평균 가중치 (최종 분석 대상) ===")
print(a.groupby('sex')[W].mean().round(0))
a['age_grp'] = pd.cut(a['age'], [19, 29, 39, 49, 59, 69, 200],
                      labels=['20-29', '30-39', '40-49', '50-59', '60-69', '70+'])
print(a.groupby('age_grp', observed=True)[W].mean().round(0))

print("\n=== 연령 구성: 비가중 vs 가중 (%) ===")
un = a['age_grp'].value_counts(normalize=True).sort_index() * 100
wt = a.groupby('age_grp', observed=True)[W].sum() / a[W].sum() * 100
print(pd.DataFrame({'비가중': un.round(1), '가중': wt.round(1)}))
print("60세 이상: 비가중 %.1f / 가중 %.1f" % (un[['60-69', '70+']].sum(), wt[['60-69', '70+']].sum()))
print("20-39세:   비가중 %.1f / 가중 %.1f" % (un[['20-29', '30-39']].sum(), wt[['20-29', '30-39']].sum()))

print("\n=== 연령대별 교육수준 분포 (가중 %) ===")
edu_lab = {1: '초졸 이하', 2: '중졸', 3: '고졸', 4: '대졸 이상'}
a2 = a[a['edu'].notna()]
tab = a2.pivot_table(index='age_grp', columns='edu', values=W, aggfunc='sum', observed=True)
print((tab.div(tab.sum(axis=1), axis=0) * 100).round(1).rename(columns=edu_lab))

print("\n=== 왜도 (비가중 / 가중) ===")
def wskew(x, w):
    m = np.average(x, weights=w)
    m2 = np.average((x - m) ** 2, weights=w)
    m3 = np.average((x - m) ** 3, weights=w)
    return m3 / m2 ** 1.5
for v in ['HE_wc', 'age']:
    print(v, "비가중 %.2f" % stats.skew(a[v]), "가중 %.2f" % wskew(a[v].to_numpy(), a[W].to_numpy()))
    for grp in [0, 1]:
        s = a[a['smoke_grp'] == grp]
        print("  smoke_grp=%d 비가중 %.2f 가중 %.2f" % (grp, stats.skew(s[v]), wskew(s[v].to_numpy(), s[W].to_numpy())))

print("\n=== 결측 비율 (분석 대상 기준, %) ===")
for v in ['incm', 'edu', 'pa_aerobic']:
    print(v, round(a[v].isna().mean() * 100, 1))
print("BD1_11 = 9 또는 결측 (분석 대상):", int((~a['BD1_11'].isin([1, 2, 3, 4, 5, 6, 8])).sum()))

print("\n=== 공식 현재흡연 정의와 비교 (BS1_1 코드는 이용지침서 p.42에서 확인) ===")
print(pd.crosstab(a['BS1_1'], a['BS3_1'], margins=True))

=== 평균 가중치 (최종 분석 대상) ===
sex
1.0    8616.0
2.0    6457.0
Name: wt_itvex, dtype: float64
age_grp
20-29     9431.0
30-39    10781.0
40-49     8380.0
50-59     7768.0
60-69     6029.0
70+       4813.0
Name: wt_itvex, dtype: float64

=== 연령 구성: 비가중 vs 가중 (%) ===
          비가중    가중
age_grp            
20-29    11.6  14.8
30-39    10.9  15.9
40-49    15.6  17.7
50-59    19.0  20.0
60-69    21.7  17.7
70+      21.2  13.8
60세 이상: 비가중 42.9 / 가중 31.5
20-39세:   비가중 22.5 / 가중 30.8

=== 연령대별 교육수준 분포 (가중 %) ===
edu      초졸 이하    중졸    고졸  대졸 이상
age_grp                          
20-29      NaN   0.5  51.4   48.1
30-39      0.2   0.7  23.6   75.5
40-49      0.4   2.4  29.8   67.5
50-59      2.4   6.9  41.5   49.3
60-69     21.3  20.5  35.4   22.8
70+       51.8  17.7  20.0   10.5

=== 왜도 (비가중 / 가중) ===
HE_wc 비가중 0.34 가중 0.38
  smoke_grp=0 비가중 0.39 가중 0.46
  smoke_grp=1 비가중 0.05 가중 0.06
age 비가중 -0.27 가중 0.03
  smoke_grp=0 비가중 -0.31 가중 0.01
  smoke_grp=1 비가중 -0.16 가중 0.02

=== 결측 비율 (분석 대상 기준, %) ===


In [ ]:
# [셀 9] 문서(docs/10_table1.md)와 노트북 출력 자동 대조
# 아래 경로를 문서 파일 위치에 맞게 수정하세요. 표의 각 행을 문자열로 비교합니다.
import os
DOC_PATH = '/content/drive/MyDrive/hdac_data/10_table1.md'

out_dir = '/content/drive/MyDrive/hdac_data'
with open(os.path.join(out_dir, 'table1_B2_weighted.md'), 'w', encoding='utf-8') as f:
    f.write(md_w)
with open(os.path.join(out_dir, 'table1_B3_unweighted.md'), 'w', encoding='utf-8') as f:
    f.write(md_u)

if os.path.exists(DOC_PATH):
    doc_lines = set(l.strip() for l in open(DOC_PATH, encoding='utf-8'))
    bad = 0
    for name, text in [('B-2 가중', md_w), ('B-3 비가중', md_u)]:
        for line in text.split('\n')[2:]:
            if line.strip() not in doc_lines:
                bad += 1
                print(f"[불일치] {name}: {line}")
    print("불일치 행 수:", bad, "(0이면 문서의 두 표가 노트북 출력과 모두 일치)")
else:
    print("문서 파일을 찾을 수 없습니다. DOC_PATH를 확인하세요.")

문서 파일을 찾을 수 없습니다. DOC_PATH를 확인하세요.
